### Refer this website for Chunk Visualization
- [Click Here](https://www.chunkviz.com/)

### Character Text Splitter

In [1]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [2]:
%pwd

'd:\\Tipto\\Generative-AI\\Text Spliters'

In [3]:
import os 
os.chdir('../')

In [4]:
from pathlib import Path
pdf_file_path = Path("Document Loaders\dl-curriculum.pdf")
pdf_doc = PyPDFLoader(pdf_file_path)

In [5]:
pdf_content = pdf_doc.load()
pdf_content[0].page_content

'CampusXDeepLearningCurriculum\nA.ArtificialNeuralNetworkandhowtoimprovethem\n1.BiologicalInspiration\n● Understandingtheneuronstructure● Synapsesandsignal transmission● Howbiological conceptstranslatetoartificial neurons\n2.HistoryofNeuralNetworks\n● Earlymodels(Perceptron)● BackpropagationandMLPs● The"AI Winter" andresurgenceof neural networks● Emergenceof deeplearning\n3.PerceptronandMultilayerPerceptrons(MLP)\n● Single-layer perceptronlimitations● XORproblemandtheneedfor hiddenlayers● MLParchitecture\n4. LayersandTheirFunctions\n● InputLayer○ Acceptinginput data● HiddenLayers○ Featureextraction● OutputLayer○ Producingfinal predictions\n5.ActivationFunctions'

In [6]:
# split based on character count
characterSplitter = CharacterTextSplitter(
    chunk_size = 100,
    length_function = len,
    chunk_overlap = 0,
    separator = '\n'
)
chunks = characterSplitter.split_text(pdf_content[0].page_content)

Created a chunk of size 118, which is longer than the specified 100
Created a chunk of size 123, which is longer than the specified 100
Created a chunk of size 107, which is longer than the specified 100


In [7]:
type(chunks)

list

**Note:**

Use split_text() when you have plain text (string) and want to split it into smaller text.
Use split_documents() when you already have LangChain Document objects(A List of documents).

In [8]:
chunks

['CampusXDeepLearningCurriculum\nA.ArtificialNeuralNetworkandhowtoimprovethem\n1.BiologicalInspiration',
 '● Understandingtheneuronstructure● Synapsesandsignal transmission● Howbiological conceptstranslatetoartificial neurons',
 '2.HistoryofNeuralNetworks',
 '● Earlymodels(Perceptron)● BackpropagationandMLPs● The"AI Winter" andresurgenceof neural networks● Emergenceof deeplearning',
 '3.PerceptronandMultilayerPerceptrons(MLP)',
 '● Single-layer perceptronlimitations● XORproblemandtheneedfor hiddenlayers● MLParchitecture',
 '4. LayersandTheirFunctions',
 '● InputLayer○ Acceptinginput data● HiddenLayers○ Featureextraction● OutputLayer○ Producingfinal predictions',
 '5.ActivationFunctions']

In [9]:
type(chunks[0])

str

In [10]:
type(pdf_content[0])

langchain_core.documents.base.Document

In [11]:
type(pdf_content[0])

langchain_core.documents.base.Document

### Recursive Character Text Splitter
---
Core Idea: The splitter follows this strategy:
1. Try splitting using large semantic boundaries<br>
2. If chunks are still too large → try smaller separators<br>
3. Continue recursively until chunks fit within chunk_size<br>

Typical separator hierarchy: ["\n\n", "\n", " ", ""].

For RecursiveCharacterTextSplitter, the chunk_size is measured in characters by default, not words or sentences.

chunk_size = 100 => Each chunk can contain up to 100 characters.
The algorithm does not stop splitting just because a separator occurs before 100 characters.

Instead it follows this logic:

Split using the current separator

Combine pieces until chunk_size would be exceeded

If a piece itself is larger than chunk_size, then recursively split it using the next smaller separator

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 20
)

In [13]:
text = """
Machine learning is a branch of artificial intelligence.

It focuses on building models that learn patterns from data.

These models can make predictions or decisions.
"""

chunks = splitter.split_text(text)

for c in chunks:
    print(c)

Machine learning is a branch of artificial intelligence.
It focuses on building models that learn patterns from data.
These models can make predictions or decisions.


### Document Structure–Based Text Splitters
---
A Document Structure Splitter divides text according to the logical structure of the document, rather than by characters or separators. The splitting is guided by format-specific structure such as:

- headings

- sections

- paragraphs

- HTML tags

- Markdown headers

- code blocks

This approach preserves semantic boundaries defined by the document format.

| Document Type       | Structure                 |
| ------------------- | ------------------------- |
| PDF research papers | sections, subsections     |
| Markdown            | `#`, `##`, `###` headings |
| HTML                | `<h1>`, `<h2>`, `<p>`     |
| Code                | classes, functions        |


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [15]:
splitter = RecursiveCharacterTextSplitter.from_language(
    language = Language.PYTHON,
    chunk_size = 300,
    chunk_overlap = 0
)

In [16]:
code = """  
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature = 0.7
)

model = ChatHuggingFace(llm = llm)
result = model.invoke("Who is the CEO of OpenAI?")
print(result.content)
"""


In [17]:
result = splitter.split_text(code)
result

['import os\nfrom dotenv import load_dotenv\nfrom langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint\n\nload_dotenv()\n\nllm = HuggingFaceEndpoint(\n    repo_id="meta-llama/Llama-3.1-8B-Instruct",\n    task="text-generation",\n    max_new_tokens=512,\n    temperature = 0.7\n)',
 'model = ChatHuggingFace(llm = llm)\nresult = model.invoke("Who is the CEO of OpenAI?")\nprint(result.content)']

In [18]:
for i in result:
    print(" - " * 20)
    print(i)

 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature = 0.7
)
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
model = ChatHuggingFace(llm = llm)
result = model.invoke("Who is the CEO of OpenAI?")
print(result.content)


### Semantic Text Splitter

A Semantic Text Splitter divides text based on meaning (semantic similarity) rather than characters, separators, or document structure.
Semantic splitting works differently from character/recursive splitting — instead of splitting on separators or character counts, it embeds sentences and splits where the meaning changes significantly.

---
**How it works internally**
1. Splits text into individual sentences
2. Embeds each sentence using an embedding model
3. Computes cosine similarity between adjacent sentences
4. Splits at points where similarity drops below a threshold (semantic shift)
---


Instead of asking:

“Where should I cut based on characters or paragraphs?”

it asks:

“Where does the topic or meaning change?”

This produces chunks that contain coherent ideas, which improves embedding quality and retrieval accuracy in RAG systems.

Pipeline:<br>
Text => Sentence segmentation => Embeddings => Similarity calculation => Semantic boundary detection => Chunks

In [19]:
%pip -q install langchain_experimental

Note: you may need to restart the kernel to use updated packages.


In [26]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

In [23]:
import os 
from dotenv import load_dotenv
load_dotenv()

HUGGINGFACE_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACE_API_TOKEN

In [27]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
splitter = SemanticChunker(
    embeddings = embeddings,
    breakpoint_threshold_type = "percentile",
    breakpoint_threshold_amount = 85 # split at top 15% sharpest drops
)
chunks = splitter.split_documents(pdf_content)

In [29]:
chunks

[Document(metadata={'producer': 'Skia/PDF m131 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Deep Learning Curriculum', 'source': 'Document Loaders\\dl-curriculum.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}, page_content='CampusXDeepLearningCurriculum\nA.ArtificialNeuralNetworkandhowtoimprovethem\n1.BiologicalInspiration\n● Understandingtheneuronstructure● Synapsesandsignal transmission● Howbiological conceptstranslatetoartificial neurons\n2.HistoryofNeuralNetworks\n● Earlymodels(Perceptron)● BackpropagationandMLPs● The"AI Winter" andresurgenceof neural networks● Emergenceof deeplearning\n3.PerceptronandMultilayerPerceptrons(MLP)\n● Single-layer perceptronlimitations● XORproblemandtheneedfor hiddenlayers● MLParchitecture\n4. LayersandTheirFunctions\n● InputLayer○ Acceptinginput data● HiddenLayers○ Featureextraction● OutputLayer○ Producingfinal predictions\n5.ActivationFunctions'),
 Document(metadata={'producer': 'Skia/PDF m131 Google Docs Renderer'

In [30]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content[:100])
    print()

Chunk 1 (658 chars):
CampusXDeepLearningCurriculum
A.ArtificialNeuralNetworkandhowtoimprovethem
1.BiologicalInspiration
●

Chunk 2 (771 chars):
● SigmoidFunction○ Characteristicsandlimitations● HyperbolicTangent(tanh)○ Comparisonwithsigmoid● Re

Chunk 3 (719 chars):
● StochasticGradientDescent(SGD)○ Advantagesinlargedatasets● Mini-BatchGradientDescent○ Balancingbet

Chunk 4 (579 chars):
○ Decidingdepthandwidth● Techniques:○ Gridsearch○ RandomSearch○ Bayesianoptimization
13.Vanishingand

Chunk 5 (628 chars):
3.ConvolutionOperation
● UnderstandingKernels/Filters○ Edgedetectionfilters○ Featureextraction● Math

Chunk 6 (603 chars):
7.LossFunctions
● Cross-EntropyLossfor Classification● MeanSquaredError for Regression
8.CNNArchitec

Chunk 7 (568 chars):
○ Layers, activations● Contributions○ Handwrittendigit recognition
12.AlexNet
● Breakthroughs○ Deepe

Chunk 8 (568 chars):
● Optimizationsfor MobileDevices
17.Pre-trainedModels&TransferLearning
● UsingModelsTrainedonImageNe

Chunk 9 (695 cha